# LexiGen Kaggle Benchmark

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
import kaggle_benchmarks as kbench

os.environ["RENDER_SUBRUNS"] = "False"


## 1. Load the dataset

Each dataset row contains:

- `train`: a list of example translation pairs
- `test`: a single held-out translation pair

The benchmark converts each task into a prompt and keeps the gold translation for scoring.


In [ ]:
TRANSLATION_TOKEN_RE = re.compile(r"[a-z]+")
DATASET_FILE = "/kaggle/input/datasets/majkel1337/isl-dataset/isl_dataset.json"


def load_task_dataset(base_path: str | Path = "."):
    base_path = Path(base_path)
    file_path = base_path / DATASET_FILE
    if file_path.exists():
        return json.loads(file_path.read_text(encoding="utf-8"))

    raise FileNotFoundError(
        "Missing LexiGen dataset file. Checked the following path:\n"
        f"{file_path}"
    )


def build_translation_prompt(train_examples, test_sentence):
    examples_text = "\n".join(
        f"{example['source']} -> {example['target']}"
        for example in train_examples
    )
    return (
        "You translate English sentences into an invented sign language.\n"
        "Infer the translation rules only from the examples.\n"
        "Return only the translated sentence as lowercase tokens separated by spaces.\n"
        "Do not add explanations, punctuation, quotes, or labels.\n\n"
        f"Examples:\n{examples_text}\n\n"
        f"Sentence: {test_sentence}\n"
        "Translation:"
    )


def build_dataset_frame(tasks):
    rows = []
    for task_index, task in enumerate(tasks, start=1):
        train_examples = [
            {"source": source, "target": target}
            for source, target in task["train"]
        ]
        test_sentence, expected_translation = task["test"][0]
        context_document = "\n".join(
            f"{example['source']} -> {example['target']}"
            for example in train_examples
        )
        rows.append(
            {
                "task_index": task_index,
                "full_prompt": build_translation_prompt(train_examples, test_sentence),
                "user_request": test_sentence,
                "context_document": context_document,
                "expected_translation": expected_translation,
            }
        )
    return pd.DataFrame(rows)


def normalize_translation(text):
    return " ".join(TRANSLATION_TOKEN_RE.findall((text or "").lower()))


tasks = load_task_dataset()
examples = build_dataset_frame(tasks)
examples[["task_index", "user_request", "expected_translation"]].head()


## 2. Define the benchmark tasks

The benchmark uses a row-level task for single examples and a batch task with signature `(llm, df) -> float` to compute overall accuracy.


In [ ]:
def extract_llm_text(response):
    if response is None:
        return ""

    if isinstance(response, str):
        return response

    if isinstance(response, dict):
        # Try common text-bearing fields used by different providers.
        for key in (
            "text",
            "content",
            "output_text",
            "response",
            "completion",
            "answer",
            "generated_text",
            "message",
        ):
            if key in response:
                candidate = extract_llm_text(response.get(key))
                if candidate:
                    return candidate

        for key in ("choices", "output", "results", "candidates"):
            items = response.get(key)
            if isinstance(items, list) and items:
                candidate = extract_llm_text(items[0])
                if candidate:
                    return candidate

        return str(response)

    if isinstance(response, list):
        parts = [extract_llm_text(item) for item in response]
        return " ".join(part for part in parts if part)

    for attr in ("text", "content", "message"):
        if hasattr(response, attr):
            candidate = extract_llm_text(getattr(response, attr))
            if candidate:
                return candidate

    return str(response)


@kbench.task(name="solve_translation", version=1)
def solve_translation(
    llm,
    full_prompt: str,
    expected_translation: str,
    task_index: int,
) -> dict:
    raw_response = None
    error = None
    try:
        raw_response = llm.prompt(full_prompt)
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"

    predicted_translation = normalize_translation(extract_llm_text(raw_response))
    expected = normalize_translation(expected_translation)

    return {
        "task_index": int(task_index),
        "is_correct": bool(predicted_translation) and predicted_translation == expected,
        "prediction": predicted_translation,
        "gold": expected,
        "response_was_empty": not bool(predicted_translation),
        "error": error,
    }


@kbench.task(name="lexigen_translation_benchmark", version=1)
def lexigen_translation_benchmark(llm, df) -> float:
    with kbench.client.enable_cache():
        runs = solve_translation.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=20,
        )

    eval_df = runs.as_dataframe()
    is_correct_series = eval_df["result"].apply(
        lambda result: bool(result.get("is_correct")) if isinstance(result, dict) else False
    )
    accuracy = float(is_correct_series.mean()) if len(is_correct_series) else 0.0
    print(f"Overall accuracy: {accuracy:.1%}")
    return accuracy

## 3. Run the evaluation

This runs the batch benchmark task on the prepared dataframe and reports overall exact-match accuracy.


In [ ]:
evaluation_columns = ["full_prompt", "expected_translation", "task_index"]

overall_accuracy = lexigen_translation_benchmark.run(
    kbench.llm,
    examples[evaluation_columns],
)
print(f"Evaluated {len(examples[evaluation_columns])} tasks")

## 4. Select the benchmark output

Kaggle Benchmarks uses `%choose` to mark the benchmark artifact that should be kept as the notebook output.


In [ ]:
%choose lexigen_translation_benchmark
